# 03 - Sliding Window RAG

**Requires GROQ_API_KEY.**

Fixed paragraph splits can cut important context in half. A sliding window
creates chunks that overlap, so no sentence is ever lost at a boundary.

```
Text:    [  chunk 1  ]
                  [  chunk 2  ]
                          [  chunk 3  ]
         <------>
          stride
```

Overlapping chunks mean that even if a sentence starts near the end of one
chunk, it also appears near the beginning of the next. The retriever always
finds it with full context.

### What you will learn

| Step | Concept |
|------|---------|
| 1 | Build a sliding window chunker with configurable size and stride |
| 2 | Index chunks with BM25 |
| 3 | Retrieve the top-k chunks for a query |
| 4 | Pass the retrieved chunks to Groq for a grounded answer |
| 5 | Compare answers with 1 chunk vs 3 chunks of context |

> Requires: GROQ_API_KEY in a .env file

## 1. Install and Import

In [ ]:
# !pip install rank-bm25 langchain-groq python-dotenv

In [ ]:
import re, os
from pathlib import Path
from collections import Counter
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

load_dotenv()
print("GROQ_API_KEY set:", bool(os.getenv("GROQ_API_KEY")))

## 2. Sliding Window Chunker

In [ ]:
def sliding_window_chunks(text, source, window=400, stride=200, min_len=100):
    """
    Split text into overlapping windows of approximately `window` words,
    advancing `stride` words at a time.

    window : target chunk size in words
    stride : how many words to advance before starting the next chunk
             (overlap = window - stride words)
    """
    words  = text.split()
    chunks = []
    start  = 0

    while start < len(words):
        end   = min(start + window, len(words))
        chunk = " ".join(words[start:end])
        if len(chunk) >= min_len:
            chunks.append({
                "source": source,
                "start_word": start,
                "end_word": end,
                "text": chunk
            })
        if end == len(words):
            break
        start += stride

    return chunks


# Demonstrate on one document
sample_text = (Path("../data") / "ancient_civilizations.txt").read_text()
demo_chunks = sliding_window_chunks(sample_text, "ancient_civilizations.txt",
                                    window=100, stride=50)

print(f"Document words : {len(sample_text.split())}")
print(f"Sliding chunks : {len(demo_chunks)}")
print(f"\nChunk 0 (words 0-100) starts with:")
print(" ", demo_chunks[0]["text"][:120])
print(f"\nChunk 1 (words 50-150) starts with:")
print(" ", demo_chunks[1]["text"][:120])

## 3. Index All Documents with BM25

In [ ]:
DATA_DIR = Path("../data")

STOP_WORDS = {
    "a", "an", "the", "is", "it", "in", "on", "at", "to", "of",
    "and", "or", "but", "for", "with", "by", "from", "as", "was",
    "are", "were", "be", "been", "has", "have", "had", "this", "that",
    "these", "those", "its", "their", "they", "also", "which", "into",
    "more", "than", "about", "such", "can", "not", "over", "up", "out"
}

def tokenize(text):
    text = re.sub(r"[^a-z0-9\s]", " ", text.lower())
    return [t for t in text.split() if t not in STOP_WORDS and len(t) > 2]


# Build sliding window chunks for all four documents
all_chunks = []
for filepath in sorted(DATA_DIR.glob("*.txt")):
    text   = filepath.read_text()
    chunks = sliding_window_chunks(text, filepath.name, window=150, stride=75)
    all_chunks.extend(chunks)
    print(f"  {filepath.name:35s}  {len(chunks)} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")

# Build BM25 index
tokenized = [tokenize(c["text"]) for c in all_chunks]
bm25      = BM25Okapi(tokenized)
print("BM25 index built")

## 4. Retrieval Function

In [ ]:
def retrieve(query, top_k=3):
    """Return top_k chunks by BM25 score."""    scores = bm25.get_scores(tokenize(query))
    ranked = sorted(zip(scores, all_chunks), key=lambda x: x[0], reverse=True)
    return [chunk for _, chunk in ranked[:top_k]]


# Quick test
results = retrieve("How did the Romans build roads?", top_k=2)
for i, r in enumerate(results, 1):
    print(f"[{i}] {r['source']}  words {r['start_word']}-{r['end_word']}")
    print(f"    {r['text'][:150]}...")
    print()

## 5. Set Up Groq LLM

In [ ]:
llm = ChatGroq(model="llama3-8b-8192", temperature=0.2)

SYSTEM_PROMPT = (
    "You are a factual assistant. Answer the question using only "
    "the provided context. If the answer is not in the context, "
    "say so clearly. Keep answers concise and accurate."
)

def answer_query(query, top_k=3):
    """Retrieve relevant chunks and generate a grounded answer."""    chunks  = retrieve(query, top_k=top_k)
    context = "\n\n".join(
        f"[Source: {c['source']}]\n{c['text']}" for c in chunks
    )

    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f"Context:\n{context}\n\nQuestion: {query}")
    ]

    response = llm.invoke(messages)
    return response.content.strip(), chunks

print("Groq LLM ready!")

## 6. Ask Questions

In [ ]:
def ask(query, top_k=3):
    print(f"Question: {query}")
    print("-" * 55)
    answer, chunks = answer_query(query, top_k)
    print(f"Answer:\n{answer}")
    print(f"\nRetrieved from: {[c['source'] for c in chunks]}")
    print()

ask("What did the ancient Sumerians contribute to civilization?")

In [ ]:
ask("What is the role of methane in climate change?")

In [ ]:
ask("How does backpropagation train a neural network?")

In [ ]:
ask("What makes the Amazon River unusual?")

## 7. Effect of Chunk Overlap

Let us compare answers when using a narrow stride (high overlap) vs a wide
stride (low overlap) on the same query.

In [ ]:
# Rebuild with two different stride settings on one document
text = (Path("../data") / "climate_science.txt").read_text()

chunks_high_overlap = sliding_window_chunks(text, "climate.txt", window=120, stride=40)
chunks_low_overlap  = sliding_window_chunks(text, "climate.txt", window=120, stride=100)

print(f"High overlap (stride=40)  : {len(chunks_high_overlap)} chunks")
print(f"Low overlap  (stride=100) : {len(chunks_low_overlap)} chunks")
print()

# Show that boundary content appears in two chunks with high overlap
BOUNDARY_WORD = 120
for chunk in chunks_high_overlap:
    if chunk["start_word"] <= BOUNDARY_WORD <= chunk["end_word"]:
        print(f"High-overlap chunk covering word {BOUNDARY_WORD}:")
        print(" ", chunk["text"][:100])
        print()
        break

## 8. Key Takeaways

| Setting | Trade-off |
|---------|-----------|
| Large window, large stride | Few chunks, fast, may miss context at boundaries |
| Large window, small stride | More overlap, better context coverage, more storage |
| Small window, any stride | Precise retrieval, but each chunk covers less ground |

**Rule of thumb for a knowledge base of long articles:**
- Window: 100-200 words
- Stride: half the window (50 percent overlap)

**Next notebook:** BM25 retrieves many candidates, then an LLM reranks them
to pick the most relevant ones before generating the answer.